# Transport Floor Logic

**Scope clarification:** NOT about training phases or benchmark evaluations. 

**Floor concept:** Data sorting floor - where signals land based on their characteristics.

**Transport:** The routing mechanism that moves data from input → scoring → floor assignment.

## Architecture Overview

```
Input Data → [Parallel Condition Evaluation] → [Weighted Scoring] → [Floor Assignment]
                    ↓                              ↓                    ↓
              Condition Set A              Experience Weights      Hook Functions
              Condition Set B              (from long usage)        (safe registry)
              Condition Set C
```

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Callable, Any
import asyncio
from concurrent.futures import ThreadPoolExecutor

# Reference the existing validation harness
ROOT = Path('c:/Users/USER/CascadeProjects')
MASTER_YAML = ROOT / 'glimpse.master.yaml'
VALIDATION_REPORT = ROOT / 'tmp/jupyter-notebook/validation-report.json'

---

## Outline: Conditions → Hooked Function Calls

### Phase 1: Condition Sketch (Parallel Evaluation)

| Condition Set | Trigger Keywords | Hook Function | Weight (Experience) |
|---------------|------------------|---------------|---------------------|
| Signal Signature | frequency, amplitude, phase, resonance | `signal_signature_score()` | 0.40 |
| Growth Pattern | parent, child, root, leaf, depth | `growth_pattern_score()` | 0.35 |
| Semantic Proximity | semantic, proximity, related, similar | `semantic_proximity()` | 0.25 |

### Phase 2: Specific Calls and Arguments

```python
# Parallel evaluation with experience weights
results = await parallel_evaluate(
    conditions=[
        ('signal_signature', dataset, {'weights': {'coherence': 0.4, 'spatial': 0.3}}),
        ('growth_pattern', dataset, {'weights': {'branching_factor': 0.5, 'depth_coverage': 0.3}}),
        ('semantic_proximity', dataset, {'threshold': 0.7})
    ],
    bias_weights={'signal': 1.35, 'growth': 1.2, 'semantic': 1.1}  # From long experience
)
```

In [ ]:
@dataclass
class TransportCondition:
    """A condition that triggers transport to a specific floor."""
    name: str
    triggers: list[str]  # Keywords that activate this condition
    hook_function: str   # Name of function in safe registry
    weight: float        # Experience-based weight
    scope: list[str]     # Where this condition applies
    route_to: list[str]  # Target floors (views)

@dataclass  
class TransportFloor:
    """A sorting floor where data lands based on transport evaluation."""
    name: str
    description: str
    bias: float         # Floor bias from experience
    conditions: list[TransportCondition] = field(default_factory=list)

# === CONDITION DEFINITIONS (from experience) ===

CONDITIONS = [
    TransportCondition(
        name='signal_signature',
        triggers=['frequency', 'amplitude', 'phase', 'resonance', 'decay', 'waveform', 'oscillation', 'harmonic'],
        hook_function='signal_signature_score',
        weight=0.40,  # Highest - signal detection is most reliable from experience
        scope=['dataset', 'entity', 'relation'],
        route_to=['flow', 'constellation']
    ),
    TransportCondition(
        name='growth_pattern',
        triggers=['parent', 'child', 'root', 'leaf', 'depth', 'level', 'ancestor', 'descendant', 'sibling', 'hierarchy'],
        hook_function='growth_pattern_score',
        weight=0.35,  # Strong - tree structures are common in data
        scope=['dataset', 'relation'],
        route_to=['flow', 'clusters']
    ),
    TransportCondition(
        name='semantic_proximity',
        triggers=['semantic', 'proximity', 'related', 'similar', 'close', 'near', 'distance'],
        hook_function='semantic_proximity',
        weight=0.25,  # Moderate - semantic matching is useful but noisy
        scope=['dataset', 'entity'],
        route_to=['constellation', 'clusters']
    ),
    TransportCondition(
        name='temporal_distance',
        triggers=['time', 'date', 'before', 'after', 'duration', 'interval', 'sequence', 'order'],
        hook_function='temporal_distance',
        weight=0.30,  # Moderate-high - temporal patterns are reliable
        scope=['relation'],
        route_to=['timeline', 'flow']
    ),
    TransportCondition(
        name='influence_link',
        triggers=['influence', 'affects', 'causes', 'triggers', 'depends', 'requires', 'enables'],
        hook_function='influence_link',
        weight=0.28,  # Moderate - causal links are valuable but sparse
        scope=['dataset', 'relation'],
        route_to=['flow', 'constellation']
    ),
]

# === FLOOR DEFINITIONS ===

FLOORS = [
    TransportFloor(
        name='flow',
        description='Pathways, branches, signal chains — directional movement through data',
        bias=1.35,  # Most commonly useful from experience
        conditions=[c for c in CONDITIONS if 'flow' in c.route_to]
    ),
    TransportFloor(
        name='constellation',
        description='Network topology — nodes and edges, gravitational clusters',
        bias=1.30,
        conditions=[c for c in CONDITIONS if 'constellation' in c.route_to]
    ),
    TransportFloor(
        name='clusters',
        description='Grouping by proximity or similarity — categorical aggregation',
        bias=1.20,
        conditions=[c for c in CONDITIONS if 'clusters' in c.route_to]
    ),
    TransportFloor(
        name='timeline',
        description='Temporal ordering — roots in time, seasonal accumulation',
        bias=1.10,
        conditions=[c for c in CONDITIONS if 'timeline' in c.route_to]
    ),
]

---

## Parallel Evaluation Engine

The transport mechanism evaluates all conditions in parallel, then applies experience-based weights.

### Logic Flow:

1. **Scan** all field names/values for trigger keywords
2. **Fire** conditions whose triggers match (parallel)
3. **Score** each fired condition using its hook function
4. **Weight** scores by experience-derived biases
5. **Route** to floor with highest combined score

In [ ]:
class TransportEngine:
    """
    Parallel transport evaluation with experience-weighted scoring.
    
    Insights from various angles:
    - Signal detection is most reliable (highest weight)
    - Tree structures are common but often incomplete (moderate weight)
    - Semantic matching is useful but noisy (lower weight)
    - Temporal patterns are reliable when present (moderate-high)
    - Causal links are valuable but sparse (moderate)
    """
    
    def __init__(self, conditions: list[TransportCondition], floors: list[TransportFloor]):
        self.conditions = conditions
        self.floors = {f.name: f for f in floors}
        self._registry: dict[str, Callable] = {}  # Hook function registry
    
    def register_hook(self, name: str, func: Callable):
        """Register a safe function for condition evaluation."""
        self._registry[name] = func
    
    def _scan_triggers(self, data: dict) -> dict[str, list[str]]:
        """Scan data for trigger keywords, return matches per condition."""
        matches = {}
        text_content = json.dumps(data).lower()
        
        for cond in self.conditions:
            found = [t for t in cond.triggers if t in text_content]
            if found:
                matches[cond.name] = found
        return matches
    
    async def evaluate_parallel(
        self, 
        data: dict,
        preset_bias: dict[str, float] = None
    ) -> dict:
        """
        Evaluate all conditions in parallel, apply experience weights.
        
        Returns transport decision with evidence trail.
        """
        # Phase 1: Scan for triggers
        trigger_matches = self._scan_triggers(data)
        
        # Phase 2: Parallel condition evaluation
        async def eval_condition(cond_name: str, triggers: list[str]) -> tuple:
            cond = next(c for c in self.conditions if c.name == cond_name)
            hook = self._registry.get(cond.hook_function)
            
            if hook:
                score = await asyncio.to_thread(hook, data, triggers)
            else:
                # Fallback: count matches
                score = len(triggers) * cond.weight
            
            return cond_name, score, triggers
        
        # Run all matched conditions in parallel
        tasks = [
            eval_condition(name, triggers) 
            for name, triggers in trigger_matches.items()
        ]
        results = await asyncio.gather(*tasks) if tasks else []
        
        # Phase 3: Apply experience weights and floor routing
        floor_scores = {name: 0.0 for name in self.floors}
        evidence = []
        
        for cond_name, base_score, triggers in results:
            cond = next(c for c in self.conditions if c.name == cond_name)
            
            # Apply preset bias if provided
            bias = (preset_bias or {}).get(cond_name, 1.0)
            weighted_score = base_score * cond.weight * bias
            
            # Distribute to target floors
            for floor_name in cond.route_to:
                floor_scores[floor_name] += weighted_score
            
            evidence.append({
                'condition': cond_name,
                'triggers': triggers,
                'base_score': base_score,
                'weighted_score': weighted_score,
                'routed_to': cond.route_to
            })
        
        # Phase 4: Apply floor biases and select destination
        for floor_name, floor in self.floors.items():
            floor_scores[floor_name] *= floor.bias
        
        # Select floor with highest score
        destination = max(floor_scores.items(), key=lambda x: x[1]) if floor_scores else (None, 0)
        
        return {
            'destination_floor': destination[0],
            'floor_score': destination[1],
            'all_floor_scores': floor_scores,
            'evidence': evidence,
            'conditions_fired': len(results),
            'triggers_matched': trigger_matches
        }

# Create engine instance
engine = TransportEngine(CONDITIONS, FLOORS)

---

## Hook Functions (Safe Registry)

These are the specific function calls that conditions hook into. Each returns a score.

### Function Signatures:

```python
def signal_signature_score(data: dict, triggers: list[str]) -> float
def growth_pattern_score(data: dict, triggers: list[str]) -> float
def semantic_proximity(data: dict, triggers: list[str]) -> float
def temporal_distance(data: dict, triggers: list[str]) -> float
def influence_link(data: dict, triggers: list[str]) -> float
```

In [ ]:
# === HOOK FUNCTION IMPLEMENTATIONS ===
# These mirror the safe functions from glimpse-logic-validation-harness

def signal_signature_score(data: dict, triggers: list[str]) -> float:
    """
    Score based on acoustic signal parameters.
    
    Experience insight: Signal keywords are highly predictive of 
    time-series/waveform data that benefits from flow visualization.
    
    Weights from long experience:
    - coherence: 0.4 (most important - signal integrity)
    - spatial: 0.3 (distribution matters)
    - reliability: 0.2 (consistency over time)
    - connectivity: 0.1 (relationship to other signals)
    """
    weights = {'coherence': 0.4, 'spatial': 0.3, 'reliability': 0.2, 'connectivity': 0.1}
    
    # Count trigger matches, weighted by importance
    trigger_weights = {
        'frequency': 1.0, 'amplitude': 1.0, 'phase': 0.9, 'resonance': 0.95,
        'decay': 0.85, 'waveform': 0.9, 'oscillation': 0.8, 'harmonic': 0.85
    }
    
    score = sum(trigger_weights.get(t, 0.5) for t in triggers)
    return score * weights['coherence']  # Primary weight


def growth_pattern_score(data: dict, triggers: list[str]) -> float:
    """
    Score based on branching/tree indicators.
    
    Experience insight: Tree structures are common but often incomplete.
    Partial trees still benefit from flow visualization.
    
    Weights from experience:
    - branching_factor: 0.5 (primary indicator)
    - depth_coverage: 0.3 (how complete the tree is)
    - connectivity: 0.2 (cross-links)
    """
    weights = {'branching_factor': 0.5, 'depth_coverage': 0.3, 'connectivity': 0.2}
    
    trigger_weights = {
        'parent': 1.0, 'child': 1.0, 'root': 0.95, 'leaf': 0.9,
        'depth': 0.85, 'level': 0.8, 'ancestor': 0.75, 'descendant': 0.75,
        'sibling': 0.7, 'hierarchy': 0.85
    }
    
    score = sum(trigger_weights.get(t, 0.5) for t in triggers)
    return score * weights['branching_factor']


def semantic_proximity(data: dict, triggers: list[str]) -> float:
    """
    Score based on semantic similarity indicators.
    
    Experience insight: Semantic matching is useful but noisy.
    Lower weight because false positives are common.
    """
    trigger_weights = {
        'semantic': 1.0, 'proximity': 0.9, 'related': 0.7, 'similar': 0.8,
        'close': 0.6, 'near': 0.5, 'distance': 0.7
    }
    
    score = sum(trigger_weights.get(t, 0.4) for t in triggers)
    return score * 0.8  # Dampen due to noise


def temporal_distance(data: dict, triggers: list[str]) -> float:
    """
    Score based on temporal indicators.
    
    Experience insight: Temporal patterns are reliable when present.
    Time-ordered data almost always benefits from timeline view.
    """
    trigger_weights = {
        'time': 1.0, 'date': 1.0, 'before': 0.9, 'after': 0.9,
        'duration': 0.85, 'interval': 0.8, 'sequence': 0.95, 'order': 0.9
    }
    
    score = sum(trigger_weights.get(t, 0.5) for t in triggers)
    return score * 0.9  # High reliability


def influence_link(data: dict, triggers: list[str]) -> float:
    """
    Score based on causal/influence indicators.
    
    Experience insight: Causal links are valuable but sparse.
    When present, they strongly indicate flow view.
    """
    trigger_weights = {
        'influence': 1.0, 'affects': 0.95, 'causes': 1.0, 'triggers': 0.9,
        'depends': 0.85, 'requires': 0.8, 'enables': 0.75
    }
    
    score = sum(trigger_weights.get(t, 0.5) for t in triggers)
    return score * 0.85  # Moderate-high, but sparse


# Register hooks with engine
engine.register_hook('signal_signature_score', signal_signature_score)
engine.register_hook('growth_pattern_score', growth_pattern_score)
engine.register_hook('semantic_proximity', semantic_proximity)
engine.register_hook('temporal_distance', temporal_distance)
engine.register_hook('influence_link', influence_link)

---

## Test Transport with Sample Data

Run the transport engine on sample datasets to verify floor assignment logic.

In [ ]:
# Sample datasets for transport testing
SAMPLE_DATASETS = {
    'audio_signal': {
        'frequency': 440,
        'amplitude': 0.8,
        'phase': 0.5,
        'resonance': 'high',
        'decay_rate': 0.3,
        'waveform': 'sine'
    },
    'org_hierarchy': {
        'parent': 'CEO',
        'child': 'VP',
        'root': 'Board',
        'depth': 5,
        'hierarchy_level': 2
    },
    'event_sequence': {
        'timestamp': '2026-03-08',
        'sequence': ['start', 'process', 'end'],
        'duration': 3600,
        'before': 'checkpoint',
        'after': 'completion'
    },
    'influence_network': {
        'influence': 'strong',
        'causes': ['A', 'B'],
        'depends_on': 'C',
        'triggers': 'cascade'
    }
}

async def test_transport():
    """Run transport on all sample datasets."""
    results = {}
    for name, data in SAMPLE_DATASETS.items():
        result = await engine.evaluate_parallel(data)
        results[name] = result
        
        print(f"\n=== {name} ===")
        print(f"Destination Floor: {result['destination_floor']}")
        print(f"Floor Score: {result['floor_score']:.3f}")
        print(f"Conditions Fired: {result['conditions_fired']}")
        print(f"All Floor Scores: {result['all_floor_scores']}")
        print(f"Triggers Matched: {result['triggers_matched']}")
    
    return results

# Run test
transport_results = await test_transport()

---

## Cross-Validation with Existing Harness

Compare transport results with the glimpse-logic-validation-harness to ensure consistency.

In [ ]:
# Load existing validation report for comparison
if VALIDATION_REPORT.exists():
    existing_report = json.loads(VALIDATION_REPORT.read_text(encoding='utf-8'))
    
    print("=== Existing Validation Report ===")
    print(f"Generated: {existing_report.get('generatedAt', 'N/A')}")
    print(f"YAML Path: {existing_report.get('yamlPath', 'N/A')}")
    
    # Compare sample runs
    existing_runs = existing_report.get('sampleRuns', [])
    print(f"\nExisting Sample Runs: {len(existing_runs)}")
    
    for run in existing_runs:
        print(f"\n  Dataset: {run.get('dataset')}")
        print(f"    Primary Lens: {run.get('primaryLens')}")
        print(f"    Top Views: {run.get('topViews')}")
        print(f"    Fired Rules: {len(run.get('firedRules', []))}")
else:
    print("No existing validation report found. Run bootstrap_glimpse_logic.mjs first.")

---

## Summary: Transport Floor Logic

### Key Insights from Experience:

1. **Signal detection** is the most reliable transport indicator (weight: 0.40)
2. **Growth patterns** are common but often incomplete (weight: 0.35)
3. **Temporal patterns** are reliable when present (weight: 0.30)
4. **Influence links** are valuable but sparse (weight: 0.28)
5. **Semantic proximity** is useful but noisy (weight: 0.25)

### Floor Biases (from long usage):

- **Flow**: 1.35 (most commonly useful)
- **Constellation**: 1.30 (network visualization)
- **Clusters**: 1.20 (grouping patterns)
- **Timeline**: 1.10 (temporal ordering)

### Parallel Evaluation Benefits:

- All conditions evaluated simultaneously
- No priority bias in evaluation order
- Combined scores provide richer signal
- Experience weights applied after evaluation

In [ ]:
# Export transport report for comparison
report_path = ROOT / 'tmp/jupyter-notebook/transport-floor-report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)

transport_report = {
    'generatedAt': '2026-03-08T21:30:00Z',
    'scope': 'Data sorting floor - NOT training phases or benchmarks',
    'conditions': [
        {
            'name': c.name,
            'triggers': c.triggers,
            'hook': c.hook_function,
            'weight': c.weight,
            'routes_to': c.route_to
        }
        for c in CONDITIONS
    ],
    'floors': [
        {
            'name': f.name,
            'description': f.description,
            'bias': f.bias
        }
        for f in FLOORS
    ],
    'sampleRuns': {
        name: {
            'destination_floor': result['destination_floor'],
            'floor_score': result['floor_score'],
            'conditions_fired': result['conditions_fired'],
            'evidence_count': len(result['evidence'])
        }
        for name, result in transport_results.items()
    }
}

report_path.write_text(json.dumps(transport_report, indent=2), encoding='utf-8')
print(f"Report exported to: {report_path}")